In [ ]:
# Notebook de mantenimiento: limpia tablas del pipeline.
#
# MODO 1 - por planta (full_reset = False):
#   Borra filas de las tablas indicadas para las plantas dadas.
#
# MODO 2 - reset total (full_reset = True):
#   Trunca TODAS las tablas del pipeline. Usar con precaucion.
#
# fab job run -P plant_codes:string=FRA
# fab job run -P full_reset:string=True

plant_codes = ""      # coma-separado: "FRA,PL2"
tables      = ""      # coma-separado. Vacio = scada_raw,scada_proc,raw_pvsyst,proc_pvsyst,pr_results
full_reset  = False   # True -> trunca TODAS las tablas ignorando plant_codes


In [ ]:
ALL_TABLES         = ['scada_raw', 'scada_proc', 'raw_pvsyst', 'proc_pvsyst',
                      'pr_results', 'ingested_files', 'pending_runs']
PARTITIONED_TABLES = ['scada_raw', 'scada_proc', 'raw_pvsyst', 'proc_pvsyst', 'pr_results']

def _exists(tbl):
    return spark.catalog.tableExists(tbl)

def _count(tbl):
    return spark.sql(f'SELECT COUNT(*) AS n FROM {tbl}').first()['n']

def _vacuum(tbl):
    try:
        spark.sql(f'VACUUM {tbl} RETAIN 168 HOURS')
    except Exception as _e:
        print(f'  VACUUM {tbl}: {_e}')

_cleaned = []

if str(full_reset).strip().upper() in ('TRUE', 'VERDADERO', '1', 'SI', 'YES'):
    print('MODO: full_reset=True -> truncando todas las tablas')
    for _tbl in ALL_TABLES:
        if not _exists(_tbl):
            print(f'  {_tbl}: no existe, omito')
            continue
        _before = _count(_tbl)
        spark.sql(f'DELETE FROM {_tbl} WHERE 1=1')
        _vacuum(_tbl)
        print(f'  {_tbl}: {_before:,} filas eliminadas')
        _cleaned.append(_tbl)
else:
    _plants = [p.strip() for p in str(plant_codes).split(',') if p.strip()]
    _tbls   = [t.strip() for t in str(tables).split(',') if t.strip()] or PARTITIONED_TABLES
    if not _plants:
        raise ValueError('plant_codes vacio. Pon al menos una planta o activa full_reset=True.')
    print(f'MODO: by_plant | plantas={_plants} | tablas={_tbls}')
    for _tbl in _tbls:
        if not _exists(_tbl):
            print(f'  {_tbl}: no existe, omito')
            continue
        if _tbl not in PARTITIONED_TABLES:
            print(f'  {_tbl}: sin plant_code, usa full_reset=True')
            continue
        for _p in _plants:
            _before = spark.sql(f"SELECT COUNT(*) AS n FROM {_tbl} WHERE plant_code = '{_p}'").first()['n']
            spark.sql(f"DELETE FROM {_tbl} WHERE plant_code = '{_p}'")
            print(f'  {_tbl}: {_p} -> {_before:,} filas eliminadas')
        _vacuum(_tbl)
        _cleaned.append(_tbl)


In [ ]:
import json, notebookutils
notebookutils.notebook.exit(json.dumps({'cleaned_tables': _cleaned}))
